In [3]:
import pandas as pd
df = pd.read_csv('dataset/hotel_booking.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 36 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

In [4]:
# Identify missing values
df.isna().sum()

hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                              16340
company         

In [5]:
# Investigate variable 'children'
df.loc[df['children'].isna(),['adults','children','babies']]

,adults,children,babies
40600,2,NaN,0
40667,2,NaN,0
40679,3,NaN,0
41160,2,NaN,0


In [6]:
# Fill in missing values for 'children'
df['children'] = df['children'].fillna(0)

# Verify missing values resolved for 'children'
df['children'].isna().sum()

np.int64(0)

In [7]:
# Investigate variable 'country'
df.loc[
    df['country'].isna(),
    ['hotel','country','market_segment','distribution_channel']]

,hotel,country,market_segment,distribution_channel
30,Resort Hotel,NaN,Direct,Direct
4127,Resort Hotel,NaN,Offline TA/TO,TA/TO
7092,Resort Hotel,NaN,Corporate,Corporate
7860,Resort Hotel,NaN,Direct,Direct
8779,Resort Hotel,NaN,Corporate,Corporate
...,...,...,...,...
65908,City Hotel,NaN,Complementary,Corporate
65909,City Hotel,NaN,Complementary,Corporate
65910,City Hotel,NaN,Complementary,Corporate
80830,City Hotel,NaN,Groups,TA/TO


In [8]:
# Fill in missing values for 'country'
df['country'] = df['country'].fillna('Unknown')

# Verify missing values resolved for 'children'
df['country'].isna().sum()

np.int64(0)

In [9]:
# Investigate variable 'agent'
df.loc[df['agent'].isna(),'distribution_channel'].value_counts()

distribution_channel
Direct       7625
Corporate    5543
TA/TO        3167
Undefined       3
GDS             2
Name: count, dtype: int64

In [10]:
columns = [
    'distribution_channel',
    'market_segment',
    'customer_type',
    'hotel'
]

for col in columns:
    print("\n")
    print(df.loc[df['agent'].isna(),col].value_counts())



distribution_channel
Direct       7625
Corporate    5543
TA/TO        3167
Undefined       3
GDS             2
Name: count, dtype: int64


market_segment
Direct           6007
Corporate        4594
Groups           4148
Complementary     640
Offline TA/TO     389
Online TA         348
Aviation          212
Undefined           2
Name: count, dtype: int64


customer_type
Transient          11729
Transient-Party     4499
Group                 99
Contract              13
Name: count, dtype: int64


hotel
Resort Hotel    8209
City Hotel      8131
Name: count, dtype: int64


In [11]:
pd.crosstab(
    df.loc[df['agent'].isna(), 'market_segment'],
    df.loc[df['agent'].isna(), 'distribution_channel']
)

distribution_channel,Corporate,Direct,GDS,TA/TO,Undefined
market_segment,,,,,
Aviation,204,0,0,8,0
Complementary,80,542,0,18,0
Corporate,4105,167,0,322,0
Direct,77,5763,1,165,1
Groups,1045,1079,0,2024,0
Offline TA/TO,1,4,0,384,0
Online TA,31,70,1,246,0
Undefined,0,0,0,0,2


In [12]:
# Investigate conflicting subset
subset = df.loc[
    (df['market_segment'] == 'Direct') &
    (df['distribution_channel'] == 'TA/TO') &
    (df['agent'].isna())
    ]

subset['company'].isna().value_counts()

company
True     158
False      7
Name: count, dtype: int64

In [13]:
pd.crosstab(
    df.loc[df['agent'].isna(),'distribution_channel'],
    df.loc[df['agent'].isna(),'company'].isna()
)

company,False,True
distribution_channel,,
Corporate,4951,592
Direct,784,6841
GDS,0,2
TA/TO,845,2322
Undefined,0,3


In [14]:
# Investigate how many agents associate w one company
df.groupby('company')['agent'].nunique().sort_values(ascending=False).head(10)

company
281.0    4
478.0    1
40.0     1
45.0     1
439.0    1
485.0    1
444.0    1
417.0    1
419.0    1
421.0    1
Name: agent, dtype: int64

In [15]:
# Investigate comp 281
df.loc[df['company'] == 281, 'agent'].value_counts(dropna=False)

agent
185.0    78
334.0    28
NaN      18
328.0     9
326.0     5
Name: count, dtype: int64

In [16]:
# How many companies does each agent work w/
df.groupby('agent')['company'].nunique().sort_values(ascending=False).head(10)

agent
9.0      9
250.0    6
14.0     4
240.0    4
290.0    2
184.0    2
328.0    1
326.0    1
314.0    1
334.0    1
Name: company, dtype: int64

In [17]:
df[['agent','company']].notna().all(axis=1).sum()

np.int64(217)

In [18]:
# Investigate missing company IDs by market segment
pd.crosstab(
    df['company'].isna(),
    df['market_segment']
)

market_segment,Aviation,Complementary,Corporate,Direct,Groups,Offline TA/TO,Online TA,Undefined
company,,,,,,,,
False,212,313,4471,208,1391,106,96,0
True,25,430,824,12398,18420,24113,56381,2


In [19]:
# Count duplicate rows
df.duplicated().sum()

np.int64(0)

In [20]:
# Investigate bookings with zero recorded nights
df['total_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']

df.loc[
    (df['total_nights']) == 0,
    ['stays_in_weekend_nights','stays_in_week_nights','reservation_status']
    ].value_counts()

stays_in_weekend_nights  stays_in_week_nights  reservation_status
0                        0                     Check-Out             680
                                               Canceled               22
                                               No-Show                13
Name: count, dtype: int64

In [21]:
df.loc[
    (df['total_nights'] == 0) &
    (df['reservation_status'] == 'Check-Out'),
    ['hotel','adults','children','babies','adr']
    ].describe(include='all')

,hotel,adults,children,babies,adr
count,680,680.000000,680.000000,680.000000,680.0
unique,2,NaN,NaN,NaN,NaN
top,Resort Hotel,NaN,NaN,NaN,NaN
freq,372,NaN,NaN,NaN,NaN
mean,NaN,1.572059,0.072059,0.007353,0.0
std,NaN,0.680541,0.333388,0.085496,0.0
min,NaN,0.000000,0.000000,0.000000,0.0
25%,NaN,1.000000,0.000000,0.000000,0.0
50%,NaN,2.000000,0.000000,0.000000,0.0
75%,NaN,2.000000,0.000000,0.000000,0.0


In [22]:
df.loc[
    (df['total_nights'] == 0) &
    (df['reservation_status'] == 'Check-Out'),
    'market_segment'
].value_counts()

market_segment
Online TA        297
Offline TA/TO    146
Direct           132
Corporate         51
Groups            35
Complementary     14
Aviation           5
Name: count, dtype: int64

In [23]:
df['scheduled_arrival_date'] = pd.to_datetime(
    df['arrival_date_year'].astype(str) + " " 
     + df['arrival_date_month'] + " " 
     + df['arrival_date_day_of_month'].astype(str),
    format="%Y %B %d"
)

df['reservation_status_date'] = pd.to_datetime(
    df['reservation_status_date']
)

df['date_difference'] = (
    df['reservation_status_date'] - df['scheduled_arrival_date']
).dt.days

# Investigate bookings with no recorded total nights but status = 'Check-Out'
df.loc[
    ((df['total_nights']) == 0) &
    (df['reservation_status'] == 'Check-Out'),
    'date_difference'
].value_counts()

date_difference
0    680
Name: count, dtype: int64

In [24]:
# Count total zero-guest bookings
df['total_guests'] = (
    df['adults'] + df['children'] + df['babies']
)

# Count bookings with zero guests
(df['total_guests'] == 0).sum()

np.int64(180)

In [25]:
# Investigate other booking details
df.loc[
    df['total_guests'] == 0,
    ['market_segment','adr','stays_in_weekend_nights','stays_in_week_nights']
    ].describe(include='all')

,market_segment,adr,stays_in_weekend_nights,stays_in_week_nights
count,180,180.000000,180.000000,180.000000
unique,7,NaN,NaN,NaN
top,Online TA,NaN,NaN,NaN
freq,69,NaN,NaN,NaN
mean,NaN,10.456500,1.288889,3.233333
std,NaN,31.681635,2.348157,5.608354
min,NaN,0.000000,0.000000,0.000000
25%,NaN,0.000000,0.000000,0.000000
50%,NaN,0.000000,0.000000,1.000000
75%,NaN,0.000000,2.000000,4.000000


In [26]:
# Investigate other booking details
((df['total_guests'] == 0) &
   (df['adr'] > 0) & 
    (df['reservation_status'] == 'Check-Out')).sum()

np.int64(31)

In [27]:
# Investigate bookings with negative adr
df.loc[df['adr'] < 0,
    ['reservation_status','adr']
    ]

,reservation_status,adr
14969,Check-Out,-6.38


In [28]:
df.loc[
    df['adr'] == 0,
    ['market_segment']
    ].value_counts()

market_segment
Complementary     680
Online TA         367
Offline TA/TO     332
Groups            252
Direct            240
Corporate          82
Aviation            6
Name: count, dtype: int64

In [29]:
# Investigate bookings with recorded total nights but adr = 0
df.loc[
    (df['adr'] == 0) &
    ((df['total_nights']) > 0) &
    (df['reservation_status'] == 'Check-Out'),
    'market_segment'
].value_counts()

market_segment
Complementary    574
Groups           193
Offline TA/TO    146
Direct            77
Online TA         50
Corporate         26
Name: count, dtype: int64

In [30]:
# Investigate unusually high adr
df['adr'].describe()

count    119390.000000
mean        101.831122
std          50.535790
min          -6.380000
25%          69.290000
50%          94.575000
75%         126.000000
max        5400.000000
Name: adr, dtype: float64

In [31]:
# Determine outliers
Q1 = df['adr'].quantile(0.25)
Q3 = df['adr'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

((df['adr'] < lower_bound) |(df['adr'] > upper_bound)).sum()

np.int64(3793)

In [32]:
df.loc[
    (df['adr'] >= lower_bound) &
    (df['adr'] <= upper_bound),
    'adr'
].mean()

np.float64(97.0366429924652)

In [33]:
# Investigate top 10 unusually high adr bookings
df.loc[
    df['adr'] > upper_bound,
    ['hotel', 'adr', 'reserved_room_type',
     'market_segment', 'arrival_date_month',
     'reservation_status']
].sort_values(by='adr', ascending=False).head(10)

,hotel,adr,reserved_room_type,market_segment,arrival_date_month,reservation_status
48515,City Hotel,5400.00,A,Offline TA/TO,March,Canceled
111403,City Hotel,510.00,A,Offline TA/TO,May,Check-Out
15083,Resort Hotel,508.00,A,Corporate,July,Check-Out
103912,City Hotel,451.50,E,Direct,December,Check-Out
13142,Resort Hotel,450.00,A,Online TA,August,Canceled
13391,Resort Hotel,437.00,H,Direct,August,Canceled
39155,Resort Hotel,426.25,G,Direct,August,Check-Out
39568,Resort Hotel,402.00,H,Online TA,August,Check-Out
39118,Resort Hotel,397.38,H,Direct,July,Check-Out
39517,Resort Hotel,392.00,G,Online TA,August,Check-Out


In [34]:
# Investigate reservation status of adr > upper_bound
df.loc[
    df['adr'] > upper_bound,
    'reservation_status'].value_counts()

reservation_status
Check-Out    2324
Canceled     1446
No-Show        23
Name: count, dtype: int64

In [35]:
# Investigate completed bookings with adr > upper_bound
df.loc[
    (df['adr'] > upper_bound) &
    (df['reservation_status'] == 'Check-Out'),
    ['hotel', 'adr', 'reserved_room_type',
     'market_segment', 'arrival_date_month']
    ].sort_values(by='adr',ascending=False).head(10)

,hotel,adr,reserved_room_type,market_segment,arrival_date_month
111403,City Hotel,510.00,A,Offline TA/TO,May
15083,Resort Hotel,508.00,A,Corporate,July
103912,City Hotel,451.50,E,Direct,December
39155,Resort Hotel,426.25,G,Direct,August
39568,Resort Hotel,402.00,H,Online TA,August
39118,Resort Hotel,397.38,H,Direct,July
39517,Resort Hotel,392.00,G,Online TA,August
39543,Resort Hotel,388.00,H,Online TA,August
39177,Resort Hotel,387.00,G,Direct,August
39960,Resort Hotel,383.00,H,Online TA,August


In [36]:
# Investigate bookings where date diff < 0
df.loc[
    df['date_difference'] < 0,
    'reservation_status'
    ].value_counts()

reservation_status
Canceled    42136
No-Show         1
Name: count, dtype: int64

In [37]:
# Validate cancelation status consistency
pd.crosstab(df['reservation_status'],df['is_canceled'])

is_canceled,0,1
reservation_status,,
Canceled,0,43017
Check-Out,75166,0
No-Show,0,1207


In [38]:
# Validate categorical fields
columns = [
    'distribution_channel',
    'market_segment',
    'customer_type',
    'hotel',
    'meal',
    'reserved_room_type',
    'assigned_room_type',
    'deposit_type'
]

for col in columns:
    print("\n")
    print(col)
    print(df[col].value_counts(dropna=False))



distribution_channel
distribution_channel
TA/TO        97870
Direct       14645
Corporate     6677
GDS            193
Undefined        5
Name: count, dtype: int64


market_segment
market_segment
Online TA        56477
Offline TA/TO    24219
Groups           19811
Direct           12606
Corporate         5295
Complementary      743
Aviation           237
Undefined            2
Name: count, dtype: int64


customer_type
customer_type
Transient          89613
Transient-Party    25124
Contract            4076
Group                577
Name: count, dtype: int64


hotel
hotel
City Hotel      79330
Resort Hotel    40060
Name: count, dtype: int64


meal
meal
BB           92310
HB           14463
SC           10650
Undefined     1169
FB             798
Name: count, dtype: int64


reserved_room_type
reserved_room_type
A    85994
D    19201
E     6535
F     2897
G     2094
B     1118
C      932
H      601
P       12
L        6
Name: count, dtype: int64


assigned_room_type
assigned_room_type
A   

In [39]:
# Remove sensitive customer info
df = df.drop(columns=[
    'name',
    'email',
    'phone-number',
    'credit_card'
])

df.columns

Index(['hotel', 'is_canceled', 'lead_time', 'arrival_date_year',
       'arrival_date_month', 'arrival_date_week_number',
       'arrival_date_day_of_month', 'stays_in_weekend_nights',
       'stays_in_week_nights', 'adults', 'children', 'babies', 'meal',
       'country', 'market_segment', 'distribution_channel',
       'is_repeated_guest', 'previous_cancellations',
       'previous_bookings_not_canceled', 'reserved_room_type',
       'assigned_room_type', 'booking_changes', 'deposit_type', 'agent',
       'company', 'days_in_waiting_list', 'customer_type', 'adr',
       'required_car_parking_spaces', 'total_of_special_requests',
       'reservation_status', 'reservation_status_date', 'total_nights',
       'scheduled_arrival_date', 'date_difference', 'total_guests'],
      dtype='str')

In [40]:
# Flags
df['adr_outlier_flag'] = (
    (df['adr'] > upper_bound) | 
    (df['adr'] < lower_bound)
)

In [41]:
df['unexplained_zero_adr_flag'] = (
        (df['adr'] == 0) &
        (df['reservation_status'] == 'Check-Out') & 
        (df['total_nights'] > 0) &
        (df['market_segment'] != 'Complementary')
)

In [42]:
df['date_inconsistency_flag'] = (
    (df['date_difference'] < 0) &
    (df['reservation_status'] != 'Canceled')
)

In [43]:
df[
    ['adr_outlier_flag','unexplained_zero_adr_flag','date_inconsistency_flag']
    ].sum()

adr_outlier_flag             3793
unexplained_zero_adr_flag     492
date_inconsistency_flag         1
dtype: int64

In [44]:
df['estimated_room_revenue'] = float('nan')

df.loc[
    (df['adr'] >= 0) &
    (df['reservation_status'] == 'Check-Out') &
    (df['total_nights'] > 0),
    'estimated_room_revenue'] = df['adr'] * df['total_nights']

df['estimated_room_revenue'].describe()

count    74485.000000
mean       349.014220
std        322.616874
min          0.000000
25%        141.000000
50%        260.640000
75%        442.000000
max       7590.000000
Name: estimated_room_revenue, dtype: float64

In [45]:
df['estimated_room_revenue'].count()

np.int64(74485)

In [49]:
## Final Data Quality Validation
df.shape

(119390, 40)

In [50]:
# Verify Missing Values
df.isna().sum()

hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               0
babies                                 0
meal                                   0
country                                0
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                              16340
company         

In [51]:
# Verify duplicates after cleaning
df.duplicated().sum()

np.int64(31994)

In [ ]:
### Duplicate Validation

After removing customer identifiers and applying data cleaning,
31,994 apparent duplicate records were identified.

These records were retained because the dataset lacks a unique
booking identifier to distinguish duplicate entries from separate
reservations with identical booking characteristics.

Removing these records could underestimate booking volume
and estimated room revenue.

In [52]:
# Verify data types
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 40 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   hotel                           119390 non-null  str           
 1   is_canceled                     119390 non-null  int64         
 2   lead_time                       119390 non-null  int64         
 3   arrival_date_year               119390 non-null  int64         
 4   arrival_date_month              119390 non-null  str           
 5   arrival_date_week_number        119390 non-null  int64         
 6   arrival_date_day_of_month       119390 non-null  int64         
 7   stays_in_weekend_nights         119390 non-null  int64         
 8   stays_in_week_nights            119390 non-null  int64         
 9   adults                          119390 non-null  int64         
 10  children                        119390 non-null  float64       
 11

In [53]:
# Export cleaned dataset
df.to_csv('dataset/hotel_booking_cleaned.csv',index=False)

In [55]:
# Compare dimensions
clean_df = pd.read_csv('dataset/hotel_booking_cleaned.csv')

print(df.shape)
print(clean_df.shape)

(119390, 40)
(119390, 40)


In [57]:
# Compare column names
df.columns.equals(clean_df.columns)

True